# CHIS 2024 Shared Cleaning and Train/Test Split

This notebook prepares the agreed 2024 Adult CHIS cohort for classification (`RISK_CONTROL`) and regression (`AH5`). It preserves the existing 11,700 respondents and original train/test assignments while defining the shared base and expanded predictor sets used across the team's model notebooks.

In [1]:
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

data_path = '/content/drive/MyDrive/ml_healthcare_project/data/'
adult_file = data_path + 'ADULT.dta'

adult = pd.read_stata(
    adult_file,
    convert_categoricals=False
)

print(adult.shape)
adult.head()

Mounted at /content/drive
(24810, 897)


,aa5c,aa5g,ah37,ah44,ab1,ab17,ab40,ab41,ab18,ab43,...,rakedw71,rakedw72,rakedw73,rakedw74,rakedw75,rakedw76,rakedw77,rakedw78,rakedw79,rakedw80
0,-1.0,1.0,-1.0,-1.0,4.0,2.0,-1.0,-1.0,-1.0,-1.0,...,1629.972960,1613.175674,1642.494583,1606.006154,1629.514266,1606.204179,1632.499091,1625.656988,1643.699871,1628.678464
1,-1.0,1.0,2.0,-1.0,4.0,2.0,-1.0,-1.0,-1.0,-1.0,...,413.072843,437.843117,425.629335,434.333120,409.272107,411.870473,421.825637,421.590222,428.317894,413.763508
2,-1.0,-1.0,-1.0,1.0,2.0,2.0,-1.0,-1.0,-1.0,-1.0,...,9307.434237,8998.146150,8977.168776,9069.840845,9199.709760,8872.577681,8988.194721,9111.363369,9045.583324,9072.971092
3,-1.0,-1.0,-1.0,1.0,3.0,2.0,-1.0,-1.0,-1.0,-1.0,...,398.598083,411.907090,400.915619,406.286372,414.198026,405.547820,404.928609,414.036862,403.782480,391.926123
4,-1.0,-1.0,2.0,-1.0,2.0,2.0,-1.0,-1.0,-1.0,-1.0,...,369.540654,373.086641,374.884886,374.720633,369.230742,373.216464,372.140441,365.088406,368.637356,367.801909


## Selected variables

Official CHIS variable names are kept in the data and code so the workflow stays traceable to the data dictionary. Plain-language labels are listed below for reporting and interpretation.

In [2]:
keep_cols = [
    # ID
    "puf1y_id",

    # Targets
    "risk_control",
    "ah5",

    # Demographics
    "srage_p1",
    "srsex",
    "racehp2_p1",
    "ur_clrt4",

    # Socioeconomic
    "sreduc",
    "povll2_p1v2",
    "povll",
    "wrkst_p1",
    "stablehouse",

    # Health / behavior
    "ab1",
    "bmi_p",
    "ac212",
    "smkcur",

    # Chronic conditions
    "ab17",
    "diabetes",
    "ab29v2",
    "ab34",
    "ac6v2",

    # Insurance / access
    "instype",
    "ins12m",
    "usual",
    "ah81b",
    "ah22",
    "pc_ins",
    "aj218",

    # Survey weight
    "rakedw0"
]

missing_cols = [col for col in keep_cols if col not in adult.columns]
print("Missing requested columns:", missing_cols)

keep_cols = [col for col in keep_cols if col in adult.columns]
df = adult[keep_cols].copy()

print(df.shape)
df.head()

Missing requested columns: []
(24810, 29)


,puf1y_id,risk_control,ah5,srage_p1,srsex,racehp2_p1,ur_clrt4,sreduc,povll2_p1v2,povll,...,ab34,ac6v2,instype,ins12m,usual,ah81b,ah22,pc_ins,aj218,rakedw0
0,24022321,3.0,4.0,60.0,2.0,1.0,1.0,2.0,1.38,2.0,...,2.0,2.0,7.0,12.0,1.0,1.0,2.0,2.0,4.0,1621.476357
1,24009490,1.0,8.0,75.0,2.0,1.0,2.0,2.0,2.05,3.0,...,2.0,2.0,3.0,12.0,1.0,2.0,2.0,2.0,3.0,415.485560
2,24005238,3.0,2.0,75.0,2.0,6.0,1.0,1.0,6.00,4.0,...,2.0,2.0,4.0,12.0,1.0,2.0,2.0,2.0,4.0,8973.697056
3,24026888,3.0,1.0,60.0,1.0,6.0,4.0,2.0,0.39,1.0,...,2.0,2.0,5.0,12.0,1.0,2.0,2.0,2.0,3.0,402.938846
4,24010558,3.0,2.0,26.0,2.0,4.0,1.0,4.0,2.64,3.0,...,2.0,2.0,7.0,11.0,1.0,2.0,2.0,2.0,4.0,370.585184


## CHIS response codes

CHIS uses negative codes for different situations. Inapplicable and proxy-skipped responses are structural survey codes, while refused, don't know, and not ascertained indicate nonresponse. They are not treated as the same type of missing data.

For the selected variables, `STABLEHOUSE = -2` is a proxy-skipped response. Those respondents are retained rather than dropped.

In [3]:
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        negative = df.loc[df[col] < 0, col].value_counts().sort_index()

        if len(negative) > 0:
            print(f"\n{col}")
            print(negative)


stablehouse
stablehouse
-2.0    16
Name: count, dtype: int64

pc_ins
pc_ins
-1.0    844
Name: count, dtype: int64


In [4]:
df_clean = df.copy()

nonresponse_codes = [-7, -8, -9]

for col in df_clean.columns:
    if pd.api.types.is_numeric_dtype(df_clean[col]):
        df_clean[col] = df_clean[col].replace(
            nonresponse_codes,
            np.nan
        )

missing_summary = pd.DataFrame({
    "Missing Count": df_clean.isna().sum(),
    "Missing %": df_clean.isna().mean() * 100
})

print("Duplicate IDs:", df_clean["puf1y_id"].duplicated().sum())

missing_summary[
    missing_summary["Missing Count"] > 0
].sort_values("Missing %", ascending=False)

Duplicate IDs: 0


,Missing Count,Missing %


## Modeling cohort and targets

`RISK_CONTROL` values 1 and 2 define the shared cohort. Value 3 represents adults without a diagnosed risk factor covered by this outcome and is not part of the classification question.

For classification, `not_controlled = 1` is the positive class. For regression, the target remains the reported number of doctor visits in the past 12 months (`AH5`).

In [5]:
model_base = df_clean[
    df_clean["risk_control"].isin([1, 2])
].copy()

model_base["not_controlled"] = (
    model_base["risk_control"] == 2
).astype(int)

model_base["coverage_continuity"] = pd.cut(
    model_base["ins12m"],
    bins=[-0.1, 0, 11, 12],
    labels=[
        "No coverage",
        "Partial-year coverage",
        "Full-year coverage"
    ]
)

print("Cohort:", model_base.shape[0])
print("\nClassification target")
print(model_base["not_controlled"].value_counts().sort_index())
print(model_base["not_controlled"].value_counts(normalize=True).sort_index() * 100)

weighted_rate = np.average(
    model_base["not_controlled"],
    weights=model_base["rakedw0"]
)

print(f"\nWeighted not-controlled rate: {weighted_rate * 100:.1f}%")

print("\nCoverage continuity")
print(model_base["coverage_continuity"].value_counts())

print("\nAH5")
print(
    model_base["ah5"].describe(
        percentiles=[.5, .75, .9, .95, .99]
    )
)

Cohort: 11700

Classification target
not_controlled
0    8560
1    3140
Name: count, dtype: int64
not_controlled
0    73.162393
1    26.837607
Name: proportion, dtype: float64

Weighted not-controlled rate: 30.0%

Coverage continuity
coverage_continuity
Full-year coverage       11190
Partial-year coverage      307
No coverage                203
Name: count, dtype: int64

AH5
count    11700.000000
mean         5.679145
std          9.214558
min          0.000000
50%          4.000000
75%          6.000000
90%         12.000000
95%         20.000000
99%         40.000000
max        300.000000
Name: ah5, dtype: float64


## Insurance and healthcare-access review

Additional insurance, affordability, and healthcare-access variables are reviewed within the existing 11,700-person modeling cohort before changing the shared predictor set.

Structural CHIS skip codes are kept separate from nonresponse because several candidate questions apply only to specific insurance or healthcare-use groups. The audit checks availability, response distributions, and overlap with predictors already in the shared data. No respondents or train/test assignments are changed here.

In [6]:
candidate_cols = [
    "uninsany",
    "ins12m",
    "ah71_p1",
    "ah81b",
    "care_pv",
    "timappt",
    "ah22",
    "pc_ins",
    "aj218"
]

missing_candidates = [
    col for col in candidate_cols
    if col not in adult.columns
]

print("Missing candidate columns:", missing_candidates)

candidate_data = adult[
    ["puf1y_id"] + candidate_cols
].copy()

candidate_audit = model_base[
    [
        "puf1y_id",
        "not_controlled",
        "ah5",
        "srage_p1",
        "instype",
        "coverage_continuity",
        "usual"
    ]
].merge(
    candidate_data,
    on="puf1y_id",
    how="left",
    validate="one_to_one"
)

print("Audit cohort:", candidate_audit.shape[0])
print(
    "IDs matched:",
    candidate_audit["puf1y_id"].nunique()
)

Missing candidate columns: []
Audit cohort: 11700
IDs matched: 11700


In [7]:
structural_codes = {
    "uninsany": [-1],
    "ins12m": [],
    "ah71_p1": [-1],
    "ah81b": [],
    "care_pv": [],
    "timappt": [-1],
    "ah22": [],
    "pc_ins": [-1],
    "aj218": []
}

audit_rows = []

for col in candidate_cols:
    values = candidate_audit[col]

    structural = values.isin(
        structural_codes[col]
    )

    nonresponse = values.isin(
        [-7, -8, -9]
    )

    valid = (
        ~structural
        & ~nonresponse
        & values.notna()
    )

    audit_rows.append({
        "Variable": col,
        "Valid n": valid.sum(),
        "Valid %": valid.mean() * 100,
        "Skipped/Inapplicable n": structural.sum(),
        "Skipped/Inapplicable %": structural.mean() * 100,
        "Nonresponse n": nonresponse.sum(),
        "Nonresponse %": nonresponse.mean() * 100
    })

candidate_summary = pd.DataFrame(audit_rows)

print(candidate_summary.round(1))

   Variable  Valid n  Valid %  Skipped/Inapplicable n  Skipped/Inapplicable %  \
0  uninsany     6693     57.2                    5007                    42.8   
1    ins12m    11700    100.0                       0                     0.0   
2   ah71_p1     6095     52.1                    5605                    47.9   
3     ah81b    11700    100.0                       0                     0.0   
4   care_pv    11700    100.0                       0                     0.0   
5   timappt     4451     38.0                    7249                    62.0   
6      ah22    11700    100.0                       0                     0.0   
7    pc_ins    11414     97.6                     286                     2.4   
8     aj218    11700    100.0                       0                     0.0   

   Nonresponse n  Nonresponse %  
0              0            0.0  
1              0            0.0  
2              0            0.0  
3              0            0.0  
4              0   

In [8]:
for col in candidate_cols:
    print(f"\n{col}")
    print(
        candidate_audit[col]
        .value_counts()
        .sort_index()
    )


uninsany
uninsany
-1.0    5007
 1.0     186
 2.0     321
 3.0    6186
Name: count, dtype: int64

ins12m
ins12m
0.0       203
1.0        15
2.0        10
3.0        15
4.0        17
5.0         9
6.0        44
7.0        16
8.0        32
9.0        50
10.0       54
11.0       45
12.0    11190
Name: count, dtype: int64

ah71_p1
ah71_p1
-1.0    5605
 1.0    2412
 2.0    3683
Name: count, dtype: int64

ah81b
ah81b
1.0     1402
2.0    10298
Name: count, dtype: int64

care_pv
care_pv
1.0    9537
2.0    2163
Name: count, dtype: int64

timappt
timappt
-1.0    7249
 1.0    3494
 2.0     957
Name: count, dtype: int64

ah22
ah22
1.0    2281
2.0    9419
Name: count, dtype: int64

pc_ins
pc_ins
-1.0      286
 1.0      883
 2.0    10531
Name: count, dtype: int64

aj218
aj218
1.0     499
2.0    1565
3.0    3248
4.0    5696
5.0     692
Name: count, dtype: int64


In [9]:
print("\nUNINSANY and coverage continuity")
print(
    pd.crosstab(
        candidate_audit["uninsany"],
        candidate_audit["coverage_continuity"]
    )
)

print("\nAH71_P1 and insurance type")
print(
    pd.crosstab(
        candidate_audit["ah71_p1"],
        candidate_audit["instype"]
    )
)

print("\nTIMAPPT and usual source of care")
print(
    pd.crosstab(
        candidate_audit["timappt"],
        candidate_audit["usual"]
    )
)

print("\nPC_INS and insurance type")
print(
    pd.crosstab(
        candidate_audit["pc_ins"],
        candidate_audit["instype"]
    )
)


UNINSANY and coverage continuity
coverage_continuity  No coverage  Partial-year coverage  Full-year coverage
uninsany                                                                   
-1.0                          11                     31                4965
 1.0                         186                      0                   0
 2.0                           6                    276                  39
 3.0                           0                      0                6186

AH71_P1 and insurance type
instype  1.0  2.0   3.0  4.0   5.0   7.0  8.0  9.0
ah71_p1                                           
-1.0     332  898  2565  420  1254     0   24  112
 1.0       0   35   199    0    79  1822  277    0
 2.0       0  128   808    0   159  2436  152    0

TIMAPPT and usual source of care
usual     1.0  2.0
timappt           
-1.0     6511  738
 1.0     3387  107
 2.0      872   85

PC_INS and insurance type
instype  1.0  2.0   3.0  4.0   5.0   7.0  8.0  9.0
pc_ins              

## Variable reference

| Role | CHIS variable | Plain language |
|---|---|---|
| Classification target | `not_controlled` | Chronic risk factors not controlled |
| Regression target | `ah5` | Doctor visits in past 12 months |
| Demographic | `srage_p1` | Age |
| Demographic | `srsex` | Gender |
| Demographic / subgroup | `racehp2_p1` | Race/ethnicity |
| Geographic / subgroup | `ur_clrt4` | Urban/rural category |
| Socioeconomic | `sreduc` | Education level |
| Socioeconomic | `povll2_p1v2` | Poverty level relative to federal poverty level |
| Income subgroup | `povll` | Poverty-level category |
| Socioeconomic | `wrkst_p1` | Employment status |
| Socioeconomic | `stablehouse` | Housing stability |
| Health | `ab1` | General health |
| Health | `bmi_p` | Body mass index |
| Behavior | `ac212` | Moderate physical activity |
| Behavior | `smkcur` | Current smoking |
| Condition | `ab17` | Asthma diagnosis |
| Condition | `diabetes` | Diabetes diagnosis |
| Condition | `ab29v2` | High blood pressure diagnosis |
| Condition | `ab34` | Heart disease diagnosis |
| Condition | `ac6v2` | Stroke diagnosis |
| Insurance / subgroup | `instype` | Current insurance type |
| Insurance | `coverage_continuity` | Full-year, partial-year, or no coverage |
| Access | `usual` | Has a usual place for healthcare |
| Affordability | `ah81b` | Problems paying medical bills |
| Access | `ah22` | Delayed or did not receive needed medical care |
| Insurance access | `pc_ins` | Difficulty finding a primary-care provider who accepts insurance |
| Access | `aj218` | Ease of obtaining needed care, tests, or treatment |
| Tracking only | `puf1y_id` | CHIS respondent ID |
| Survey only | `rakedw0` | CHIS survey weight |

## Predictor types

CHIS-coded groups are treated as categorical rather than as continuous distances. `POVLL2_P1V2` and `BMI_P` remain numeric. `POVLL` is kept for the income subgroup audit, and `RAKEDW0` is kept for weighted summaries rather than used as a predictor.

In [10]:
base_categorical = [
    "srsex",
    "racehp2_p1",
    "ur_clrt4",
    "sreduc",
    "wrkst_p1",
    "stablehouse",
    "ab1",
    "ac212",
    "smkcur",
    "ab17",
    "diabetes",
    "ab29v2",
    "ab34",
    "ac6v2"
]

base_numeric = [
    "srage_p1",
    "povll2_p1v2",
    "bmi_p"
]

access_features = [
    "instype",
    "coverage_continuity",
    "usual",
    "ah81b",
    "ah22",
    "pc_ins",
    "aj218"
]

base_features = (
    base_categorical
    + base_numeric
)

expanded_features = (
    base_features
    + access_features
)

target_cols = [
    "not_controlled",
    "ah5"
]

audit_cols = [
    "puf1y_id",
    "povll",
    "rakedw0"
]

subgroup_cols = [
    "racehp2_p1",
    "povll",
    "instype",
    "ur_clrt4"
]

print("Base predictors:", len(base_features))
print("Expanded predictors:", len(expanded_features))
print("Insurance/access predictors:", len(access_features))

Base predictors: 17
Expanded predictors: 24
Insurance/access predictors: 7


In [11]:
model_cols = (
    audit_cols
    + target_cols
    + expanded_features
)

model_df = model_base[
    model_cols
].copy()

print(model_df.shape)
print("Missing values:", model_df.isna().sum().sum())
print("Duplicate IDs:", model_df["puf1y_id"].duplicated().sum())
print(
    "Proxy-skipped housing rows:",
    (model_df["stablehouse"] == -2).sum()
)

model_df.head()

(11700, 29)
Missing values: 0
Duplicate IDs: 0
Proxy-skipped housing rows: 11


,puf1y_id,povll,rakedw0,not_controlled,ah5,srsex,racehp2_p1,ur_clrt4,sreduc,wrkst_p1,...,srage_p1,povll2_p1v2,bmi_p,instype,coverage_continuity,usual,ah81b,ah22,pc_ins,aj218
1,24009490,3.0,415.485560,0,8.0,2.0,1.0,2.0,2.0,5.0,...,75.0,2.05,28.70,3.0,Full-year coverage,1.0,2.0,2.0,2.0,3.0
5,24006490,4.0,74.699387,1,20.0,1.0,6.0,3.0,4.0,5.0,...,70.0,4.99,25.95,3.0,Full-year coverage,1.0,2.0,2.0,2.0,4.0
12,24028667,4.0,96.530595,0,15.0,1.0,6.0,3.0,4.0,1.0,...,75.0,5.06,32.95,3.0,Full-year coverage,1.0,2.0,2.0,2.0,3.0
13,24007771,4.0,93.938835,0,4.0,1.0,6.0,2.0,2.0,2.0,...,70.0,4.65,25.05,3.0,Full-year coverage,1.0,2.0,2.0,2.0,2.0
15,24016506,4.0,338.681554,0,10.0,1.0,6.0,4.0,3.0,5.0,...,70.0,6.00,25.76,3.0,Full-year coverage,1.0,2.0,2.0,2.0,4.0


In [12]:
blocked_cols = {
    "risk_control",
    "ah5",
    "puf1y_id",
    "povll",
    "rakedw0",
    "ins12m",
    "uninsany",
    "ah71_p1",
    "care_pv",
    "timappt",
    "acmdnum",
    "ab150",
    "ab41",
    "ab152"
}

print(
    "Blocked columns in predictors:",
    sorted(
        set(expanded_features)
        & blocked_cols
    )
)

Blocked columns in predictors: []


## Train/test split

The original 80/20 split is preserved so every model family uses the same respondents as the earlier work. The existing training and test respondent IDs are read from the shared CSV files and merged onto the updated 24-predictor cohort.

No new random split is generated.

In [13]:
old_train = pd.read_csv(
    data_path + "CHIS_2024_train.csv",
    usecols=["puf1y_id"]
)

old_test = pd.read_csv(
    data_path + "CHIS_2024_test.csv",
    usecols=["puf1y_id"]
)

old_train["puf1y_id"] = pd.to_numeric(
    old_train["puf1y_id"]
).astype("int64")

old_test["puf1y_id"] = pd.to_numeric(
    old_test["puf1y_id"]
).astype("int64")

model_df["puf1y_id"] = pd.to_numeric(
    model_df["puf1y_id"]
).astype("int64")

print("Saved training IDs:", len(old_train))
print("Saved test IDs:", len(old_test))

print(
    "ID types:",
    old_train["puf1y_id"].dtype,
    old_test["puf1y_id"].dtype,
    model_df["puf1y_id"].dtype
)

print(
    "Duplicate training IDs:",
    old_train["puf1y_id"].duplicated().sum()
)

print(
    "Duplicate test IDs:",
    old_test["puf1y_id"].duplicated().sum()
)

Saved training IDs: 9360
Saved test IDs: 2340
ID types: int64 int64 int64
Duplicate training IDs: 0
Duplicate test IDs: 0


In [14]:
all_saved_ids = set(
    pd.concat([
        old_train["puf1y_id"],
        old_test["puf1y_id"]
    ])
)

current_ids = set(
    model_df["puf1y_id"]
)

print(
    "Cohort IDs unchanged:",
    current_ids == all_saved_ids
)

print(
    "Train/test overlap:",
    len(
        set(old_train["puf1y_id"])
        & set(old_test["puf1y_id"])
    )
)

train_df = old_train.merge(
    model_df,
    on="puf1y_id",
    how="left",
    validate="one_to_one"
)

test_df = old_test.merge(
    model_df,
    on="puf1y_id",
    how="left",
    validate="one_to_one"
)

print("Training:", train_df.shape)
print("Test:", test_df.shape)

print(
    "Missing training targets:",
    train_df["not_controlled"].isna().sum()
)

print(
    "Missing test targets:",
    test_df["not_controlled"].isna().sum()
)

Cohort IDs unchanged: True
Train/test overlap: 0
Training: (9360, 29)
Test: (2340, 29)
Missing training targets: 0
Missing test targets: 0


In [15]:
print("Training classification balance")
print(train_df["not_controlled"].value_counts().sort_index())
print(train_df["not_controlled"].value_counts(normalize=True).sort_index() * 100)

print("\nTest classification balance")
print(test_df["not_controlled"].value_counts().sort_index())
print(test_df["not_controlled"].value_counts(normalize=True).sort_index() * 100)

print("\nTraining AH5")
print(
    train_df["ah5"].describe(
        percentiles=[.5, .75, .9, .95, .99]
    )
)

print("\nTest AH5")
print(
    test_df["ah5"].describe(
        percentiles=[.5, .75, .9, .95, .99]
    )
)

Training classification balance
not_controlled
0    6848
1    2512
Name: count, dtype: int64
not_controlled
0    73.162393
1    26.837607
Name: proportion, dtype: float64

Test classification balance
not_controlled
0    1712
1     628
Name: count, dtype: int64
not_controlled
0    73.162393
1    26.837607
Name: proportion, dtype: float64

Training AH5
count    9360.000000
mean        5.722222
std         9.453288
min         0.000000
50%         4.000000
75%         6.000000
90%        12.000000
95%        20.000000
99%        40.000000
max       300.000000
Name: ah5, dtype: float64

Test AH5
count    2340.000000
mean        5.506838
std         8.189945
min         0.000000
50%         3.000000
75%         6.000000
90%        12.000000
95%        20.000000
99%        40.000000
max       150.000000
Name: ah5, dtype: float64


## Modeling inputs

Both tracks use the same respondents and compare the same two feature sets.

**Base set:** 17 demographic, socioeconomic, health, behavioral, and condition-profile predictors.

**Expanded set:** the 17 base predictors plus 7 insurance, affordability, and healthcare-access predictors.

Categorical encoding and scaling are fit inside each model notebook using training data only.

In [16]:
X_train_base = train_df[
    base_features
].copy()

X_test_base = test_df[
    base_features
].copy()

X_train_expanded = train_df[
    expanded_features
].copy()

X_test_expanded = test_df[
    expanded_features
].copy()

y_train_class = train_df[
    "not_controlled"
].copy()

y_test_class = test_df[
    "not_controlled"
].copy()

y_train_reg = train_df[
    "ah5"
].copy()

y_test_reg = test_df[
    "ah5"
].copy()

print(
    "Base:",
    X_train_base.shape,
    X_test_base.shape
)

print(
    "Expanded:",
    X_train_expanded.shape,
    X_test_expanded.shape
)

print(
    "Classification target:",
    y_train_class.shape,
    y_test_class.shape
)

print(
    "Regression target:",
    y_train_reg.shape,
    y_test_reg.shape
)

Base: (9360, 17) (2340, 17)
Expanded: (9360, 24) (2340, 24)
Classification target: (9360,) (2340,)
Regression target: (9360,) (2340,)


In [17]:
model_df.to_csv(
    data_path + "CHIS_2024_modeling_cohort.csv",
    index=False
)

train_df.to_csv(
    data_path + "CHIS_2024_train.csv",
    index=False
)

test_df.to_csv(
    data_path + "CHIS_2024_test.csv",
    index=False
)

print("Saved:")
print("CHIS_2024_modeling_cohort.csv", model_df.shape)
print("CHIS_2024_train.csv", train_df.shape)
print("CHIS_2024_test.csv", test_df.shape)

Saved:
CHIS_2024_modeling_cohort.csv (11700, 29)
CHIS_2024_train.csv (9360, 29)
CHIS_2024_test.csv (2340, 29)


In [18]:
check_cohort = pd.read_csv(
    data_path + "CHIS_2024_modeling_cohort.csv"
)

check_train = pd.read_csv(
    data_path + "CHIS_2024_train.csv"
)

check_test = pd.read_csv(
    data_path + "CHIS_2024_test.csv"
)

print("Cohort:", check_cohort.shape)
print("Training:", check_train.shape)
print("Test:", check_test.shape)

print(
    "Cohort IDs preserved:",
    set(check_cohort["puf1y_id"].astype(str))
    ==
    set(model_df["puf1y_id"].astype(str))
)

print(
    "Training IDs preserved:",
    set(check_train["puf1y_id"].astype(str))
    ==
    set(train_df["puf1y_id"].astype(str))
)

print(
    "Test IDs preserved:",
    set(check_test["puf1y_id"].astype(str))
    ==
    set(test_df["puf1y_id"].astype(str))
)

print(
    "Missing values:",
    check_cohort.isna().sum().sum()
)

Cohort: (11700, 29)
Training: (9360, 29)
Test: (2340, 29)
Cohort IDs preserved: True
Training IDs preserved: True
Test IDs preserved: True
Missing values: 0


## Modeling notes

- `not_controlled = 1` is the positive classification class.
- The original class distribution is kept for baseline modeling; resampling can be considered later only if needed.
- `AH5` remains on its original scale for the baseline regression models. Its right-skewed distribution should be considered when comparing MAE and RMSE.
- `RAKEDW0` is retained for weighted descriptive or subgroup summaries, not as a model predictor.
- Subgroup evaluation will use race/ethnicity, poverty category, insurance type, and urban/rural category.
- One-hot encoding and model-specific scaling should be learned from the training data, then applied to the test data.